<a href="https://colab.research.google.com/github/AngieIdrobo/InteligenciaArtificial/blob/main/Sistema_Transporte.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Importar las librerias e instalas astar

!pip install astar

from astar import AStar
from collections import defaultdict


# Relacionar los datos
# Se pone la base de conocimiento del sistema de transporte masivo (MIO)
# se modela el grafo y el peso es el tiempo en minutos
# las reglas representan la conexión entre las estaciones

class BaseConocimiento:
    def __init__(self):
        self.grafo = defaultdict(list)
        self.reglas = []
        self.estaciones = {}
        self._cargar_red()
        self._cargar_reglas()

    def _cargar_red(self):
        conexiones = [
            ("UniValle",      "San Bosco",     4, "T30A"),
            ("San Bosco",     "Estadio",       3, "T30A"),
            ("Estadio",       "Capri",         4, "T30A"),
            ("Capri",         "Universidades", 3, "T30A"),
            ("Universidades", "Granada",       5, "T30A"),
            ("Granada",       "Terminal",      6, "T30A"),
            ("Universidades", "Salomia",       5, "T31A"),
            ("Salomia",       "Calima",        4, "T31A"),
            ("Calima",        "Terminal",      7, "T31A"),
            ("Estadio",       "Menga",         8, "A33"),
            ("Menga",         "Centenario",    6, "A33"),
            ("Granada",       "Calima",        4, "P1"),
        ]
        for origen, destino, tiempo, linea in conexiones:
            self.grafo[origen].append((destino, tiempo, linea))
            self.grafo[destino].append((origen, tiempo, linea))

        # Se registran las coordenadas aproximadas de las estaciones

        self.estaciones = {
            "UniValle":      (3.376, -76.530),
            "San Bosco":     (3.390, -76.532),
            "Estadio":       (3.407, -76.535),
            "Capri":         (3.420, -76.537),
            "Universidades": (3.430, -76.538),
            "Salomia":       (3.445, -76.520),
            "Calima":        (3.460, -76.510),
            "Granada":       (3.435, -76.540),
            "Terminal":      (3.470, -76.515),
            "Menga":         (3.415, -76.550),
            "Centenario":    (3.408, -76.558),
        }

    def _cargar_reglas(self):
        self.reglas = {
            "penalizacion_transbordo": 5,   # R1: Regla 1 minutos extras por cambiar de línea
            "factor_hora_pico":        1.3, # R2: Regla 2 multiplicador en hora pico
            "bonificacion_directa":   -3,   # R3: Regla 3 descuento si no hay transbordos
        }

In [2]:
# Lógica de búsqueda
# Se crea la clase y se definen los métodos

class SolverMIO(AStar):

    def __init__(self, base_conocimiento, hora=8):
        self.bc = base_conocimiento
        self.hora = hora

    # comparar solo el nombre de estación al llegar al destino

    def is_goal_reached(self, current, goal):
        return current[0] == goal[0]

    # Método para retornar los nodos vecinos

    def neighbors(self, node):
        estacion, linea_actual = node
        vecinos = []
        for destino, tiempo, linea in self.bc.grafo[estacion]:
            vecinos.append((destino, linea))
        return vecinos

    # Método que trae el costo real entre dos nodos adyacentes

    def distance_between(self, n1, n2):
        estacion_origen, linea_actual = n1
        estacion_destino, linea_nueva = n2

        # Obtener tiempo base del grafo

        tiempo_base = 0
        for vecino, tiempo, linea in self.bc.grafo[estacion_origen]:
            if vecino == estacion_destino and linea == linea_nueva:
                tiempo_base = tiempo
                break

        costo = float(tiempo_base)

        # R1: penalización por transbordo

        if linea_actual is not None and linea_actual != linea_nueva:
            costo += self.bc.reglas["penalizacion_transbordo"]

        # R2: hora pico
        if 7 <= self.hora <= 9 or 17 <= self.hora <= 19:
            costo *= self.bc.reglas["factor_hora_pico"]

        return round(costo, 1)


    # Método heurística para la estimación optimista al destino

    def heuristic_cost_estimate(self, current, goal):
        estacion_actual, _ = current
        estacion_destino, _ = goal

        coords = self.bc.estaciones
        if estacion_actual not in coords or estacion_destino not in coords:
            return 0

        lat1, lon1 = coords[estacion_actual]
        lat2, lon2 = coords[estacion_destino]
        distancia = ((lat2 - lat1)**2 + (lon2 - lon1)**2) ** 0.5
        return round(distancia * 111 / 30 * 60, 1)

        # Se toma una aproximación de 1 grado ≈ 111 km, velocidad promedio 30 km/h

In [3]:
# Gestionar todo el sistema
# Se crea la clase y los métodos

class SistemaRutas:
    def __init__(self):
        self.bc = BaseConocimiento()

    # Encontrar la mejor ruta

    def encontrar_ruta(self, origen, destino, hora=8):
        if origen not in self.bc.grafo:
            return None, f"Estación '{origen}' no existe en la red."
        if destino not in self.bc.grafo:
            return None, f"Estación '{destino}' no existe en la red."

        solver = SolverMIO(self.bc, hora=hora)

        # Nodo inicio y fin: (estacion, linea)
        nodo_inicio = (origen, None)
        nodo_fin    = (destino, None)

        resultado = solver.astar(nodo_inicio, nodo_fin)

        if resultado is None:
            return None, "No se encontró ruta entre las estaciones."

        return list(resultado), None

    # Mostrar la ruta

    def mostrar_resultado(self, origen, destino, hora=8):
        ruta, error = self.encontrar_ruta(origen, destino, hora)

        print("\n" + "="*55)
        if error:
            print(f"  Error: {error}")
        else:
            estaciones    = [nodo[0] for nodo in ruta]
            lineas        = [nodo[1] for nodo in ruta if nodo[1] is not None]
            lineas_unicas = list(dict.fromkeys(lineas))

            # Calcular tiempo total

            solver = SolverMIO(self.bc, hora=hora)
            tiempo_total = sum(
                solver.distance_between(ruta[i], ruta[i+1])
                for i in range(len(ruta) - 1)
            )

            # R3: bonificación ruta directa
            transbordos = len(lineas_unicas) - 1
            if transbordos == 0:
                tiempo_total += self.bc.reglas["bonificacion_directa"]

            print(f"  RUTA ENCONTRADA")
            print(f"  Recorrido  : {' → '.join(estaciones)}")
            print(f"  Líneas     : {' → '.join(lineas_unicas)}")
            print(f"  Paradas    : {len(estaciones) - 1}")
            print(f"  Transbordos: {transbordos}")
            print(f"  Tiempo est.: {round(tiempo_total, 1)} min  (hora: {hora}h)")
        print("="*55)


In [4]:
# Pruebas
sistema = SistemaRutas()

# Ruta con transbordo, hora pico
sistema.mostrar_resultado("UniValle", "San Bosco",   hora=10)

# Ruta corta alimentador
sistema.mostrar_resultado("UniValle", "Terminal",    hora=8)

# Consulta interactiva
sistema.mostrar_resultado("Estadio",  "Centenario",  hora=15)

print("\n--- Consulta personalizada ---")
origen  = input("Estación origen : ")
destino = input("Estación destino: ")
hora    = int(input("Hora (0-23)     : "))
sistema.mostrar_resultado(origen, destino, hora)


  RUTA ENCONTRADA
  Recorrido  : UniValle → San Bosco
  Líneas     : T30A
  Paradas    : 1
  Transbordos: 0
  Tiempo est.: 1.0 min  (hora: 10h)

  RUTA ENCONTRADA
  Recorrido  : UniValle → San Bosco → Estadio → Capri → Universidades → Granada → Terminal
  Líneas     : T30A
  Paradas    : 6
  Transbordos: 0
  Tiempo est.: 29.5 min  (hora: 8h)

  RUTA ENCONTRADA
  Recorrido  : Estadio → Menga → Centenario
  Líneas     : A33
  Paradas    : 2
  Transbordos: 0
  Tiempo est.: 11.0 min  (hora: 15h)

--- Consulta personalizada ---
Estación origen : UniValle
Estación destino: Calima
Hora (0-23)     : 12

  RUTA ENCONTRADA
  Recorrido  : UniValle → San Bosco → Estadio → Capri → Universidades → Granada → Calima
  Líneas     : T30A → P1
  Paradas    : 6
  Transbordos: 1
  Tiempo est.: 28.0 min  (hora: 12h)
